In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["userId","movieId","rating","timestamp"]
)

ratings

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


Split the dataset !!!

Create ID Mappings

In [28]:
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

user_ids = train["userId"].unique()
movie_ids = train["movieId"].unique()

user_map = {id:i for i,id in enumerate(user_ids)}
movie_map = {id:i for i,id in enumerate(movie_ids)}

n_users = len(user_ids)
n_movies = len(movie_ids)

print("n_users",n_users)
print("n_movies",n_movies)

n_users 6040
n_movies 3683


Initialize Parameters

In [35]:
k = 40

P = np.random.normal(scale=0.1, size=(n_users,k))
Q = np.random.normal(scale=0.1, size=(n_movies,k))

bu = np.zeros(n_users)
bi = np.zeros(n_movies)

mu = train["rating"].mean()

SGD Training

In [36]:
lr = 0.0015
reg = 0.08
epochs = 60

for epoch in range(epochs):

    train_data = train.sample(frac=1).reset_index(drop=True)

    for row in train_data.itertuples():

        u = user_map[row.userId]
        i = movie_map[row.movieId]
        r = row.rating

        pred = mu + bu[u] + bi[i] + np.dot(P[u], Q[i])

        err = r - pred

        bu[u] += lr * (err - reg * bu[u])
        bi[i] += lr * (err - reg * bi[i])

        P[u] += lr * (err * Q[i] - reg * P[u])
        Q[i] += lr * (err * P[u] - reg * Q[i])

    print("epoch", epoch, "done")

epoch 0 done
epoch 1 done
epoch 2 done
epoch 3 done
epoch 4 done
epoch 5 done
epoch 6 done
epoch 7 done
epoch 8 done
epoch 9 done
epoch 10 done
epoch 11 done
epoch 12 done
epoch 13 done
epoch 14 done
epoch 15 done
epoch 16 done
epoch 17 done
epoch 18 done
epoch 19 done
epoch 20 done
epoch 21 done
epoch 22 done
epoch 23 done
epoch 24 done
epoch 25 done
epoch 26 done
epoch 27 done
epoch 28 done
epoch 29 done
epoch 30 done
epoch 31 done
epoch 32 done
epoch 33 done
epoch 34 done
epoch 35 done
epoch 36 done
epoch 37 done
epoch 38 done
epoch 39 done
epoch 40 done
epoch 41 done
epoch 42 done
epoch 43 done
epoch 44 done
epoch 45 done
epoch 46 done
epoch 47 done
epoch 48 done
epoch 49 done
epoch 50 done
epoch 51 done
epoch 52 done
epoch 53 done
epoch 54 done
epoch 55 done
epoch 56 done
epoch 57 done
epoch 58 done
epoch 59 done


In [37]:
def predict(user_id, movie_id):

    if user_id not in user_map or movie_id not in movie_map:
        return mu

    u = user_map[user_id]
    i = movie_map[movie_id]

    return mu + bu[u] + bi[i] + np.dot(P[u], Q[i])

In [38]:
def recommend(user_id, k=10):

    watched = train[train.userId == user_id]["movieId"].values

    scores = []

    for movie in movie_ids:

        if movie in watched:
            continue

        scores.append((movie, predict(user_id, movie)))

    scores.sort(key=lambda x: x[1], reverse=True)

    return [m for m,_ in scores[:k]]

In [39]:
def evaluate(k=10):

    precisions = []
    recalls = []

    for user in test.userId.unique():

        relevant = test[test.userId == user]["movieId"].values

        recs = recommend(user, k)

        hits = len(set(recs) & set(relevant))

        precisions.append(hits / k)
        recalls.append(hits / len(relevant))

    print("Precision@",k,":", np.mean(precisions))
    print("Recall@",k,":", np.mean(recalls))

In [40]:
evaluate(10)

Precision@ 10 : 0.06157668102020536
Recall@ 10 : 0.022652722575571084


### first result: k = 20, epoch = 20

---

Precision@ 10 : 0.08752898310698908

Recall@ 10 : 0.029743707575789785

### second result: k = 50, epoch = 80

---

Precision@ 10 : 0.05799933752898311
Recall@ 10 : 0.020643313579094446

### third result: K = 30, epochs = 50, lr = 0.002, reg = 0.05

---

Precision@ 10 : 0.07030473666777079
Recall@ 10 : 0.025039390160645457

### finaal result: K = 40, epochs = 60, lr = 0.0015, reg = 0.08

---

Precision@ 10 : 0.06157668102020536
Recall@ 10 : 0.022652722575571084

| Setting            | Result           |
| ------------------ | ---------------- |
| Low capacity model | best             |
| High capacity      | overfit          |
| Regularized model  | partial recovery |


Regular SVD works fine